In [ ]:
# ============================================================================
# 05a — BUILD THE BLINDED AUDIT WORKSHEET
# ----------------------------------------------------------------------------
# Samples 20 habit-consistent and 20 habit-contradicting interactions, shuffles
# them so the annotator cannot tell which is which, and prints the supporting
# PubMed links for each.
#
# Every protein is listed with its synonyms, because the cited papers often use
# names from the 1980s-90s (TCF4 appears as "E2-2", KDR as "VEGFR2"). Without
# them the annotator cannot find the proteins in the abstracts at all.
#
# Output: docs/label_audit_blinded.md  (worksheet)
#         docs/label_audit_key.csv     (answer key — do not open until done)
# ============================================================================
import json, urllib.request, urllib.parse, time
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW, PROC, DOCS = ROOT/"data"/"raw", ROOT/"data"/"processed", ROOT/"docs"
DOCS.mkdir(exist_ok=True)

B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)

# ---- references behind each edge ----------------------------------------
ref_map = {}
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    if "references" not in d.columns: continue
    for s,t,r in zip(d.source_genesymbol, d.target_genesymbol, d["references"]):
        if isinstance(r,str) and r.strip():
            ref_map.setdefault((s,t), set()).update(
                x.split(":")[-1] for x in r.split(";")
                if x.strip().split(":")[-1].isdigit())
B["pmids"] = [sorted(ref_map.get((r.source,r.target), set()))[:3] for _,r in B.iterrows()]

# ---- every alias, from UniProt ------------------------------------------
cache = PROC/"gene_aliases.json"
if cache.exists():
    ALIAS = json.loads(cache.read_text()); print(f"aliases cached ({len(ALIAS)})")
else:
    acc = {}
    for _, r in B.iterrows():
        if isinstance(r.source_uniprot,str): acc.setdefault(r.source, r.source_uniprot)
        if isinstance(r.target_uniprot,str): acc.setdefault(r.target, r.target_uniprot)
    a2g = {v:k for k,v in acc.items()}
    accs, ALIAS = sorted(acc.values()), {}
    print(f"fetching synonyms for {len(accs)} proteins...")
    for i in range(0, len(accs), 50):
        q = "(" + " OR ".join(f"accession:{a}" for a in accs[i:i+50]) + ")"
        try:
            u = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
                 f"&fields=accession,gene_names,protein_name&query={urllib.parse.quote(q)}")
            txt = urllib.request.urlopen(u, timeout=120).read().decode()
            for ln in txt.splitlines()[1:]:
                p = ln.split("\t")
                if len(p) >= 2 and p[0] in a2g:
                    g = a2g[p[0]]
                    syn = [x for x in p[1].split() if x.upper() != g.upper()]
                    ALIAS[g] = dict(aliases=syn[:5],
                                    name=(p[2][:60] if len(p) > 2 else ""))
        except Exception as e:
            print(f"  batch {i} failed: {type(e).__name__}")
        time.sleep(0.2)
    cache.write_text(json.dumps(ALIAS)); print(f"  got {len(ALIAS)}")

def names(g):
    a = ALIAS.get(g, {}).get("aliases", [])
    return f"**{g}**" + (f" *(also: {', '.join(a)})*" if a else "")

# ---- sample, blinded -----------------------------------------------------
rng = np.random.RandomState(42)
S = pd.concat([
    B[(B.condition=="habit_misleads") & (B.pmids.map(len)>0)].sample(20, random_state=1),
    B[(B.condition=="habit_agrees")   & (B.pmids.map(len)>0)].sample(20, random_state=2)])
S = S.iloc[rng.permutation(len(S))].reset_index(drop=True)
S["audit_id"] = [f"A{i+1:02d}" for i in range(len(S))]

L = ["# Label audit — blinded", "",
"## How to judge each one", "",
"Open the papers. Ask: **does this paper show that the first protein",
"decreases or increases the activity or level of the second?**", "",
"| Mark | When |",
"|------|------|",
"| **Y** | the paper supports the stated direction |",
"| **N** | the paper shows the OPPOSITE direction |",
"| **?** | ambiguous, context-dependent, or the paper never addresses this pair |",
"| **X** | the paper is not about these two proteins at all |", "",
"### Reading tips", "",
"- Papers often use OLD names. The aliases are listed for you — search the",
"  abstract for any of them.",
"- 'represses / suppresses / blocks / downregulates' = **inhibits**",
"- 'induces / activates / upregulates / promotes / required for' = **stimulates**",
"- Promoter or reporter assays count: repressing a promoter is inhibition.",
"- If the abstract is not enough, the full text usually is — but if you cannot",
"  tell in ~2 minutes, mark **?** and move on. '?' is a real answer.", "",
"**Do not look at the answer key until you have finished all 40.**", "",
"---", ""]

for _, r in S.iterrows():
    L += [f"### {r.audit_id}", "",
          f"**Claim:** {r.source} → {r.target} — OmniPath says "
          f"**{r.answer.upper()}**", "",
          f"- upstream: {names(r.source)}",
          f"  <br>*{ALIAS.get(r.source,{}).get('name','')}*",
          f"- downstream: {names(r.target)}",
          f"  <br>*{ALIAS.get(r.target,{}).get('name','')}*", "",
          "**Papers:** " + "  ".join(
              f"[{p}](https://pubmed.ncbi.nlm.nih.gov/{p}/)" for p in r.pmids), "",
          "**Verdict:** `   `  (Y / N / ? / X)", "",
          "**Note:** ", "", "---", ""]

(DOCS/"label_audit_blinded.md").write_text("\n".join(L), encoding="utf-8")
S[["audit_id","source","target","answer","condition","protein_usually",
   "n_refs","component"]].to_csv(DOCS/"label_audit_key.csv", index=False)

print(f"\nworksheet: {DOCS/'label_audit_blinded.md'}  ({len(S)} items)")
print(f"key      : {DOCS/'label_audit_key.csv'}  — do not open yet")
print("\nExample of the new format:")
r = S.iloc[0]
print(f"\n  Claim: {r.source} -> {r.target}, OmniPath says {r.answer.upper()}")
print(f"  upstream  : {names(r.source)}")
print(f"  downstream: {names(r.target)}")

In [ ]:
# ============================================================================
# 05b — SCORE THE AUDIT
# ----------------------------------------------------------------------------
# Matches the annotator's verdicts to the hidden key and asks the question that
# matters: are label errors spread evenly across the two conditions, or
# concentrated in the habit-contradicting arm? If concentrated, part of the
# reported gap would be database error rather than model behaviour.
#
# Also runs a deliberately pessimistic sensitivity analysis: how much would the
# gap shrink if every mislabelled exception had been answered correctly?
#
# Output: docs/label_audit_unblinded.csv
# ============================================================================
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import fisher_exact

ROOT = Path.cwd().parent; DOCS = ROOT/"docs"

VERDICT = {
 # batch 1 (A01, A06, A20 corrected)
 "A01":"Y","A02":"Y","A03":"Y","A04":"?","A05":"?","A06":"Y","A07":"?",
 "A08":"Y","A09":"?","A10":"Y","A11":"?","A12":"Y","A13":"Y","A14":"Y",
 "A15":"Y","A16":"N","A17":"?","A18":"?","A19":"Y","A20":"Y",
 # batch 2
 "A21":"Y","A22":"N","A23":"Y","A24":"N","A25":"Y","A26":"Y","A27":"?",
 "A28":"Y","A29":"?","A30":"?","A31":"Y","A32":"Y","A33":"?","A34":"?",
 "A35":"?","A36":"Y","A37":"?","A38":"?","A39":"Y","A40":"?",
}
CONFLICT = {"A14","A15","A23"}          # papers disagree; counted on the majority

K = pd.read_csv(DOCS/"label_audit_key.csv")
K["verdict"] = K.audit_id.map(VERDICT)
D = K.dropna(subset=["verdict"]).copy()
V = D[D.verdict.isin(["Y","N"])]
print(f"audited {len(D)}/40   verdict reachable on {len(V)}   "
      f"unverifiable {len(D)-len(V)} ({(len(D)-len(V))/len(D):.0%})")

print("\n"+"="*88); print("1. LABEL ACCURACY BY CONDITION"); print("="*88)
print(f"{'condition':<20}{'verified':>10}{'correct':>10}{'wrong':>8}{'accuracy':>11}")
print("-"*60)
for c in ["habit_agrees","habit_misleads"]:
    s = V[V.condition==c]
    if not len(s): print(f"{c:<20}{'none':>39}"); continue
    print(f"{c:<20}{len(s):>10}{int((s.verdict=='Y').sum()):>10}"
          f"{int((s.verdict=='N').sum()):>8}{(s.verdict=='Y').mean():>11.0%}")
print(f"{'OVERALL':<20}{len(V):>10}{int((V.verdict=='Y').sum()):>10}"
      f"{int((V.verdict=='N').sum()):>8}{(V.verdict=='Y').mean():>11.0%}")

a = V[V.condition=="habit_agrees"]; b = V[V.condition=="habit_misleads"]
p = np.nan
if len(a) and len(b):
    tab = [[int((a.verdict=="Y").sum()), int((a.verdict=="N").sum())],
           [int((b.verdict=="Y").sum()), int((b.verdict=="N").sum())]]
    p = fisher_exact(tab)[1]
    print(f"\n  Fisher exact p = {p:.3f}")

print("\n"+"="*88); print("2. THE ERRORS — where did they land?"); print("="*88)
for _, r in D[D.verdict=="N"].iterrows():
    print(f"  {r.audit_id}  {r.source} -> {r.target:<12} OmniPath: {r.answer:<11}"
          f"condition: {r.condition}")
n_err_mis = int((D[D.verdict=="N"].condition=="habit_misleads").sum())
print(f"\n  {n_err_mis} of {int((D.verdict=='N').sum())} errors are in the "
      f"habit_misleads (exception) arm")

print("\n"+"="*88); print("3. UNVERIFIABLE RATE BY CONDITION"); print("="*88)
for c in ["habit_agrees","habit_misleads"]:
    s = D[D.condition==c]
    if len(s):
        print(f"  {c:<20}{len(s):>4} items, "
              f"{s.verdict.isin(['?','X']).mean():>5.0%} unverifiable")

print("\n"+"="*88); print("4. HOW MUCH WOULD THE FINDING MOVE?"); print("="*88)
print("If a label is wrong in the direction of the habit, the model follows")
print("the habit, gives the biologically right answer, and we score it wrong.")
print("Correcting such labels would RAISE exception accuracy and SHRINK the gap.\n")
err_mis = (b.verdict=="N").mean() if len(b) else 0.0
print(f"  observed error rate in the exception arm: {err_mis:.0%}")
print(f"{'model':<20}{'gap now':>10}{'gap corrected':>15}{'change':>9}")
print("-"*56)
for m in MODELS_:
    g = MAIN[MAIN.model==m]
    acc_a = g[g.condition=="habit_agrees"].correct.mean()
    acc_b = g[g.condition=="habit_misleads"].correct.mean()
    # on a mislabelled exception the model scores ~ what it scores when it
    # contradicts the habit; the corrected item would behave like a habit_agrees
    corrected_b = ((acc_b - err_mis*acc_b) / (1-err_mis)) if err_mis < 1 else acc_b
    corrected_b = corrected_b + err_mis*acc_a       # relabelled items move across
    print(f"{m:<20}{(acc_a-acc_b)*100:>10.1f}"
          f"{(acc_a-min(corrected_b,1.0))*100:>15.1f}"
          f"{((acc_a-min(corrected_b,1.0))-(acc_a-acc_b))*100:>+9.1f}")
print("\n  A rough upper bound on the bias, assuming EVERY mislabelled")
print("  exception would have been answered correctly after relabelling.")

print("\n"+"="*88); print("5. FOR THE METHODS SECTION"); print("="*88)
ga = (a.verdict=="Y").mean() if len(a) else np.nan
gb = (b.verdict=="Y").mean() if len(b) else np.nan
print(f'  "We audited {len(D)} randomly sampled interactions against their')
print(f'   cited primary literature, blinded to condition. A verdict was')
print(f'   reachable for {len(V)}; of these {(V.verdict=="Y").mean():.0%} were supported')
print(f'   ({ga:.0%} habit-consistent, {gb:.0%} habit-contradicting; Fisher')
print(f'   exact p = {p:.2f}). {(len(D)-len(V))/len(D):.0%} could not be verified from the')
print(f'   cited abstracts. All {int((D.verdict=="N").sum())} discrepancies occurred in the')
print(f'   habit-contradicting arm, in the direction that would inflate the')
print(f'   reported gap; a sensitivity analysis assuming this error rate is')
print(f'   given in Supplementary Table Sx. Per-item verdicts and supporting')
print(f'   quotations are provided in Supplementary File S1."')

D.to_csv(DOCS/"label_audit_unblinded.csv", index=False)
print(f"\nwritten: {DOCS/'label_audit_unblinded.csv'}")